# 0. Understand Project Sessions And Routes

This notebook explains the setup that later notebooks rely on: why AutoML needs a project name, where `config.py` and project-local files are resolved from, what a `Session` contains, how the active session is saved in a Python `contextvars.ContextVar`, and when to pass `session=...` explicitly.

The short version: the project name points to `projects/<project_name>`, repo-root `.env` supplies service settings, `config.py` defines the durable recipe, `project_dir` anchors local sources such as CSV files, and the session turns those pieces plus runtime choices like dry-run/namespace into MLflow and GCS routes.

## High-level source flow

```text
Inputs we need
- project_name: selects projects/<project_name>/ and becomes part of routes;
    optional — when omitted it is inferred from your working directory
    (projects/<name>/...) or the sole project in the repo
- repo_root: repo containing projects/ and .env; usually inferred
- dry_run: routes artifacts under dry_run/... to isolate test runs
- namespace: optional route prefix for QA/team isolation
- experiment_id: optional override; otherwise comes from config.py RUN_CONFIG

automl.use_project([project_name], ...)
- resolves the project name: the explicit argument, else inferred from cwd
- resolves repo_root and project_dir = repo_root/projects/<project_name>
- loads repo_root/.env if present; shell env wins
- reads GCS_BUCKET, GCS_PREFIX, MLFLOW_TRACKING_URI into ProjectConfig
- imports project_dir/config.py and reads PROJECT_CONFIG
- creates Session(config, dry_run, namespace, experiment_id)
- saves Session in contextvars and binds MLflow/GCS routing

After setup
- implicit: data.list_datasets() uses automl.session()
- explicit: data.list_datasets(session=active) uses the passed Session
- subprocess CLI calls need flags again; contextvars stay in this Python process
```

Other env values are read where they are used: `GCP_PROJECT` by the GCS client, `SNOWFLAKE_DATABASE` / `SNOWFLAKE_SCHEMA` by the Snowflake source identity, and MLflow auth values by the MLflow client library.

In [1]:
from __future__ import annotations

import shlex

import pandas as pd
from IPython.display import display

import automl
from automl import data, experiment
from automl.mlflow import routing as mlflow_routing


## 1. `use_project` is the setup step

`automl.use_project(...)` does the setup once for this Python process:

- resolves the project name — the explicit argument, or, when you call
  `automl.use_project()` with no name, inferred from your working directory
  (under `projects/<name>/`) or the repo's sole project — and its `project_dir`;
- loads repo-root `.env` if it exists, without overriding variables already set in the shell;
- finds and imports `projects/<project_name>/config.py`;
- builds a `ProjectConfig` from the recipe in that file and environment values such as `GCS_BUCKET` and `MLFLOW_TRACKING_URI`;
- creates a `Session` with project, experiment, dry-run, and namespace state;
- stores that session in a contextvar so later library calls can find it;
- binds MLflow/GCS routing for the same session.

The name is the single key: `project_dir`, the import package, `config_path`, and
`instructions_path` all derive from it — so an inferred name and an explicit one
resolve everything the same, consistent way (no mismatch is possible).

The config is loaded when `use_project` runs. If you edit `config.py`, rerun this cell so the session sees the new recipe.

In [2]:
DRY_RUN = True
NAMESPACE = ""

# The project name is optional. Called with no name, use_project() infers it from
# this notebook's location under projects/<name>/ (this notebook lives in
# projects/example_homecredit/notebooks/, so it resolves to "example_homecredit").
# Pass it explicitly to override, e.g. automl.use_project("example_homecredit", ...).
active = automl.use_project(dry_run=DRY_RUN, namespace=NAMESPACE)
config = active.config
PROJECT_NAME = active.project_name  # the resolved name (inferred here)

display(
    {
        "project": active.project_name,
        "active_experiment_id": active.active_experiment_id,
        "dry_run": active.dry_run,
        "namespace": active.namespace or "<none>",
        "config_path": str(config.config_path),
        "instructions_path": str(config.instructions_path),
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
    }
)


{'project': 'example_homecredit',
 'active_experiment_id': 'example-homecredit',
 'dry_run': True,
 'namespace': '<none>',
 'config_path': '/Users/wendao/1.working_directory/brigit-data-science-automl-v1/projects/example_homecredit/config.py',
 'instructions_path': '/Users/wendao/1.working_directory/brigit-data-science-automl-v1/projects/example_homecredit/PROJECT_INSTRUCTIONS.md',
 'repo_root': '/Users/wendao/1.working_directory/brigit-data-science-automl-v1',
 'project_dir': '/Users/wendao/1.working_directory/brigit-data-science-automl-v1/projects/example_homecredit'}

## 2. The session is active context, not a hidden global file

After `use_project`, `automl.session()` reads the active session from a contextvar. That makes notebook cells concise while still letting reusable code pass a session explicitly.

In [3]:
current = automl.session()

display(
    {
        "automl.session() is active": current is active,
        "same_project": current.project_name == active.project_name,
        "same_experiment": current.active_experiment_id == active.active_experiment_id,
        "same_config_object": current.config is active.config,
    }
)


{'automl.session() is active': True,
 'same_project': True,
 'same_experiment': True,
 'same_config_object': True}

## 3. Calls can be implicit or explicit

Most public domain functions accept `session=None`. When omitted, they use `automl.session()` internally. Passing `session=active` is the explicit equivalent.

Use the implicit style for short notebook exploration after a clear setup cell. Use the explicit style in reusable helpers, tests, or notebooks that intentionally compare multiple sessions.

In [4]:
try:
    implicit_index = data.list_datasets()
    explicit_index = data.list_datasets(session=active)
    session_call_view = {
        "status": "storage available",
        "implicit_dataset_count": len(implicit_index.datasets),
        "explicit_dataset_count": len(explicit_index.datasets),
        "same_active_dataset": implicit_index.active_dataset_id == explicit_index.active_dataset_id,
    }
except Exception as exc:
    session_call_view = {
        "status": "storage unavailable for this kernel/env",
        "why_this_can_happen": "GCS/MLflow env may not be configured yet, or no live service is reachable.",
        "exception_type": type(exc).__name__,
        "message": str(exc),
    }

display(session_call_view)


{'status': 'storage available',
 'implicit_dataset_count': 0,
 'explicit_dataset_count': 0,
 'same_active_dataset': True}

## 4. What comes from `config.py`

The session wraps a `ProjectConfig`. The config owns the durable recipe: task, data source, eval spec, run config, split names, and model routes. The session adds runtime choices like dry-run mode, namespace, and optional experiment override.

In [5]:
run_config = config.require_run_config()

display(
    {
        "task": type(config.require_task()).__name__,
        "target_column": config.target_column,
        "experiment_id_from_config": run_config.experiment_id,
        "active_experiment_id": active.active_experiment_id,
        "train_split_name": run_config.train_split,
        "eval_split_name": run_config.eval_split,
        "primary_metric": config.primary_metric,
        "manager_route": f"{run_config.models.manager.model}/{run_config.models.manager.effort}",
        "proposer_route": f"{run_config.models.proposer.model}/{run_config.models.proposer.effort}",
        "coder_route": f"{run_config.models.coder.model}/{run_config.models.coder.effort}",
    }
)


{'task': 'BinaryClassification',
 'target_column': 'target',
 'experiment_id_from_config': 'example-homecredit',
 'active_experiment_id': 'example-homecredit',
 'train_split_name': 'train',
 'eval_split_name': 'test',
 'primary_metric': 'auc',
 'manager_route': 'opus/high',
 'proposer_route': 'opus/high',
 'coder_route': 'opus/high'}

## 5. Routes explain where MLflow/GCS artifacts land

The route is derived from session state. Dry-run and namespace are not per-call flags on data/eval/trial functions; they belong to the session established by `use_project`.

In [6]:
route_examples = pd.DataFrame(
    [
        {
            "case": "normal",
            "route": mlflow_routing.experiment_route_for(
                project_name=active.project_name,
                experiment_id=active.active_experiment_id,
                dry_run=False,
                namespace="",
            ),
        },
        {
            "case": "dry run",
            "route": mlflow_routing.experiment_route_for(
                project_name=active.project_name,
                experiment_id=active.active_experiment_id,
                dry_run=True,
                namespace="",
            ),
        },
        {
            "case": "namespace + dry run",
            "route": mlflow_routing.experiment_route_for(
                project_name=active.project_name,
                experiment_id=active.active_experiment_id,
                dry_run=True,
                namespace="qa",
            ),
        },
    ]
)

active_route = mlflow_routing.experiment_route_for(
    project_name=active.project_name,
    experiment_id=active.active_experiment_id,
    dry_run=active.dry_run,
    namespace=active.namespace,
)

display(route_examples)
display({"active_route_for_this_notebook": active_route})


,case,route
0,normal,example_homecredit/example-homecredit
1,dry run,dry_run/example_homecredit/example-homecredit
2,namespace + dry run,qa/dry_run/example_homecredit/example-homecredit


{'active_route_for_this_notebook': 'dry_run/example_homecredit/example-homecredit'}

## 6. Contextvars do not cross subprocess boundaries

The active session exists in this Python kernel. If a notebook launches the CLI with `subprocess`, that is a different process, so the command must pass route-defining flags such as `--project`, `--dry-run`, `--namespace`, or `--experiment-id` explicitly.

In [7]:
command = ["uv", "run", "automl", "--project", active.project_name]
if active.dry_run:
    command.append("--dry-run")
if active.namespace:
    command.extend(["--namespace", active.namespace])
if active.experiment_id is not None:
    command.extend(["--experiment-id", active.active_experiment_id])
command.extend(["data", "list"])

print(" ".join(shlex.quote(part) for part in command))


uv run automl --project example_homecredit --dry-run data list


## 7. Validate the setup before running anything heavy

`validate_project(live=True)` (from `automl.project` — each domain owns its
own validation recipe) checks the active session end to end:

- **Structural** — `config.py` defines `TASK`/`DATA`/`EVAL`/`RUN_CONFIG`, the required
  environment variables are set, and no scaffold `TBD_` placeholders remain
  (`project.config.*`, `project.env.*`, `project.placeholders`).
- **Connectivity** (`live=True`) — a GCS write/read/delete probe under the project
  prefix and one authenticated MLflow query (`project.connections.gcs`,
  `project.connections.mlflow`). Snowflake-backed projects get a pending warning
  until live Snowflake loading is implemented.

The same recipe backs the CLI — `uv run automl validate project` always runs the
live tier — and the `/brigit-automl:validate` skill. In the library, `live`
defaults to `False` so programmatic callers stay offline unless they opt in.

In [8]:
from automl.project import validate_project

# Validates the session bound by use_project() above. live=True adds the
# service probes; without it only the offline structural checks run.
report = validate_project(live=True)

display(report.to_json())
assert report.passed, "fix the reported issues before materializing data or running trials"

{'schema_version': 1, 'passed': True, 'issues': []}

## 8. How to read the rest of the notebooks

- Notebook 1 materializes the dataset and logs immutable artifact pointers.
- Notebook 2 profiles a logged dataset, or runs the agent-led loop.
- Notebook 3 authors a trial in the notebook and optionally runs it.
- Notebook 4 forks a prior logged trial.
- Notebook 5 reevaluates an existing model run.
- Notebook 6 inspects runs, models, datasets, predictions, and artifacts.

All of them start by establishing the same kind of session you saw here.